In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""

import json
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler  

# variables
SKIP_TARGET = "galectin","p2", "thrombin" # These targets shouldnt be in the pipeline anymore, but just to be safe.
CORR_THRESHOLD = 0.9  # correlation cutoff for feature removal
VAR_THRESHOLD = 0.001   #  variance cutoff

files = {
    "2D": "../../../2D/RANDOM/A_RF_MORGAN_COUNT/dG_pred_&_error_all_splits.json",
    "3D": "../../../3D/RANDOM/A_RF_3D_DESCRIPTORS/dG_pred_&_error_all_splits.json",
    "Pycomsia": "../../../PYC/RANDOM/B_FORMAT_PREDICTIONS/dG_pred_&_error_all_splits.json",
    "MIF": "../../../MIF/RANDOM/B_FORMAT_PREDICTIONS/dG_pred_&_error_all_splits.json"
}

def loo_ridge(alpha):
    splits = {m: json.load(open(p)) for m, p in sorted(files.items())}
    n_splits = len(next(iter(splits.values())))
    data = {}
    for m in sorted(files):
        for i in range(n_splits):
            for t, pairs in splits[m][i].items():
                if t in SKIP_TARGET: continue
                rec = data.setdefault(t, {"X": [], "y": [], "idx": []})
                for pair, v in pairs.items():
                    feats = [splits[x][i][t][pair]["prediction"] for x in sorted(files)]
                    rec["X"].append(feats)
                    rec["y"].append(v["experimental"])
                    rec["idx"].append((i, pair))
    preds_map = {}
    target_true_pred = {}
    weights_map = {}  # store model coefficients per target

    for t, rec in data.items():
        X = np.array(rec["X"]); y = np.array(rec["y"])
        # remove highly correlated features (could have used package but manual i know exactly what its doing)
        corr = np.corrcoef(X, rowvar=False)
        to_remove = set()
        for i in range(corr.shape[0]):
            for j in range(i+1, corr.shape[0]):
                if abs(corr[i, j]) > CORR_THRESHOLD:
                    to_remove.add(j)
        keep_idx = sorted(set(range(X.shape[1])) - to_remove)
        X = X[:, keep_idx]
        # remove low variance features
        var = np.var(X, axis=0)
        keep_idx = [i for i in range(len(var)) if var[i] > VAR_THRESHOLD]
        X = X[:, keep_idx]
        #
        y_true, y_pred = [], []
        weights_map[t] = {}  # initialize target-specific weights mapping
        # LOOCV over the 100 base-model splits: train on the rows of 99 splits,
        # evaluate on the held-out split's test-set predictions / experimental labels
        split_of = np.array([i for i, _ in rec["idx"]])
        for s in range(n_splits):
            te = split_of == s
            if not te.any(): continue
            tr = ~te
            scaler = StandardScaler()
            Xtr_scaled = scaler.fit_transform(X[tr])
            Xte_scaled = scaler.transform(X[te])
            model = Ridge(alpha=alpha)
            model.fit(Xtr_scaled, y[tr])
            p = model.predict(Xte_scaled)
            for k, j in enumerate(np.where(te)[0]):
                y_true.append(y[j]); y_pred.append(p[k])
                preds_map.setdefault(t, {})[rec["idx"][j]] = float(p[k])
                weights_map[t][rec["idx"][j]] = model.coef_.copy()

        target_true_pred[t] = (np.array(y_true), np.array(y_pred))

    loo_splits = []
    for i in range(n_splits):
        s = {}
        for t, pairs in splits[next(iter(splits))][i].items():
            if t == SKIP_TARGET: continue
            s[t] = {}
            for pair, v in pairs.items():
                pred = preds_map[t][(i, pair)]
                s[t][pair] = {"prediction": pred, "experimental": v["experimental"]}
        loo_splits.append(s)

    # RMSE per target
    rmse_per_target = {}
    for t, (yt, yp) in target_true_pred.items():
        mse = np.mean((yt - yp)**2)
        rmse = np.sqrt(mse)
        rmse_per_target[t] = rmse

    return rmse_per_target, loo_splits, weights_map



In [2]:
rmse_per_target , results_ridge, weights_map = loo_ridge(0.1)  # run with alpha=0.001 to get predictions


with open("dG_ridge_LOO_pred&error.json", "w") as f:
    json.dump(results_ridge, f, indent=4)